In [1]:
import sys
print(sys.executable)

import langchain
print(langchain.__version__)

d:\Users\MyPC\miniconda3\envs\env\python.exe
1.3.4


In [ ]:
import re
import sqlparse
import pandas as pd
import mysql.connector
from xgboost import XGBRegressor
from sqlparse.tokens import Keyword, DML
from sklearn.preprocessing import LabelEncoder
from sqlparse.sql import IdentifierList, Identifier
from langchain_core.prompts import ChatPromptTemplate
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings
)
from langgraph.graph import START, END, StateGraph
from langgraph.graph import END
from dotenv import load_dotenv
import os

C:\Users\MyPC\AppData\Local\Temp\ipykernel_4272\3979191941.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


In [4]:
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
llm = ChatGoogleGenerativeAI(api_key = GOOGLE_API_KEY, model = "gemini-2.5-flash",temperature = 0.1)

In [5]:
R = llm.invoke("Can you optimize SQL queries?").content
print(R)

Yes, absolutely! Optimizing SQL queries is a crucial skill for database administrators, developers, and anyone working with data. Efficient queries lead to faster application performance, reduced resource consumption (CPU, memory, I/O), and a better user experience.

Here's a comprehensive guide to optimizing SQL queries, covering various techniques and considerations:

---

## The Optimization Process

Before diving into specific techniques, it's important to follow a systematic approach:

1.  **Identify Slow Queries:**
    *   Use database monitoring tools (e.g., `pg_stat_statements` for PostgreSQL, Slow Query Log for MySQL, SQL Server Profiler).
    *   Application Performance Monitoring (APM) tools can also highlight slow database calls.
    *   Look for queries that consume a lot of time, CPU, or I/O.

2.  **Understand the Query:**
    *   What is the query trying to achieve?
    *   What tables are involved?
    *   What are the relationships between tables?
    *   What are the 

Feature-1 (SQL Analyzer)

In [6]:
class SQLAnalyzer:

    def __init__(self, query):
        self.query = query
        self.parsed = sqlparse.parse(query)[0]
        self.issues = []

    def check_select_star(self):
        if "*" in self.query:
            self.issues.append(
                "Avoid using SELECT *. Specify only required columns."
            )

    def check_missing_where(self):
        query_upper = self.query.upper()

        if query_upper.startswith("SELECT") and "WHERE" not in query_upper:
            self.issues.append(
                "No WHERE clause detected. Query may scan the entire table."
            )

    def check_too_many_joins(self, threshold=3):
        join_count = self.query.upper().count("JOIN")

        if join_count > threshold:
            self.issues.append(
                f"Query contains {join_count} JOINs. Consider simplifying joins."
            )

    def check_cartesian_join(self):
        query_upper = self.query.upper()

        if "JOIN" in query_upper and " ON " not in query_upper:
            self.issues.append(
                "Possible Cartesian Join detected (JOIN without ON condition)."
            )

    def check_function_in_where(self):
        query_upper = self.query.upper()

        functions = [
            "YEAR(",
            "MONTH(",
            "DAY(",
            "UPPER(",
            "LOWER(",
            "DATE("
        ]

        for func in functions:
            if func in query_upper:
                self.issues.append(
                    f"Function '{func[:-1]}' used in WHERE clause may prevent index usage."
                )

    def check_subquery(self):
        query_upper = self.query.upper()

        if "SELECT" in query_upper and query_upper.count("SELECT") > 1:
            self.issues.append(
                "Subquery detected. Consider replacing with JOIN if appropriate."
            )

    def analyze(self):

        self.check_select_star()
        self.check_missing_where()
        self.check_too_many_joins()
        self.check_cartesian_join()
        self.check_function_in_where()
        self.check_subquery()

        return self.issues

In [7]:
query = """
SELECT *
FROM orders o
JOIN customers c
ON o.customer_id = c.customer_id
WHERE YEAR(order_date)=2025
"""

analyzer = SQLAnalyzer(query)

issues = analyzer.analyze()

print("SQL Analysis Report")
print("-" * 50)

for i, issue in enumerate(issues, start=1):
    print(f"{i}. {issue}")

SQL Analysis Report
--------------------------------------------------
1. Avoid using SELECT *. Specify only required columns.
2. Possible Cartesian Join detected (JOIN without ON condition).
3. Function 'YEAR' used in WHERE clause may prevent index usage.


Feature-2 (Query Explanation)

In [ ]:
# Connect to MySQL
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="786Areef*",
    database="walmart_db",
    port=3306
)

cursor = conn.cursor(dictionary=True)

In [ ]:
def get_explain_plan(query):
    """
    Runs EXPLAIN on a SQL query and returns
    execution plan + performance warnings.
    """

    cursor = conn.cursor(dictionary=True)

    cursor.execute(f"EXPLAIN FORMAT=TRADITIONAL {query}")
    plan = cursor.fetchall()

    warnings = []

    for row in plan:

        table_name = row.get("table")
        access_type = row.get("type")
        rows_scanned = row.get("rows")
        index_used = row.get("key")
        extra = row.get("Extra")

        # Performance Checks
        if access_type == "ALL":
            warnings.append(
                f"Full table scan detected on '{table_name}'. Consider creating an index."
            )

        if index_used is None:
            warnings.append(
                f"No index used on '{table_name}'."
            )

        if extra:
            if "Using filesort" in extra:
                warnings.append(
                    "Filesort detected. Consider indexing ORDER BY columns."
                )

            if "Using temporary" in extra:
                warnings.append(
                    "Temporary table created. Query may be optimized."
                )

    return {
        "plan": plan,
        "warnings": list(set(warnings))
    }

In [ ]:
# Query to Analyze
query = """
SELECT *
FROM walmart_sales
WHERE Branch = 'A';
"""

# Run EXPLAIN
cursor.execute(f"EXPLAIN FORMAT=TRADITIONAL {query}")
plan = cursor.fetchall()

# Display Execution Plan
print("\nExecution Plan\n")

for row in plan:
    print(row)

# Query Metrics
print("\nQuery Metrics\n")

warnings = []

for row in plan:

    table_name = row.get("table")
    access_type = row.get("type")
    rows_scanned = row.get("rows")
    index_used = row.get("key")
    extra = row.get("Extra")

    print(f"Table: {table_name}")
    print(f"Access Type: {access_type}")
    print(f"Rows Scanned: {rows_scanned}")
    print(f"Index Used: {index_used}")
    print(f"Extra Info: {extra}")
    print("-" * 50)

    # Performance Checks

    if access_type == "ALL":
        warnings.append(
            f"Full table scan detected on '{table_name}'. Consider creating an index."
        )

    if index_used is None:
        warnings.append(
            f"No index used on '{table_name}'."
        )

    if extra:

        if "Using filesort" in extra:
            warnings.append(
                "Filesort detected. Consider indexing ORDER BY columns."
            )

        if "Using temporary" in extra:
            warnings.append(
                "Temporary table created. Query may be optimized."
            )

# Performance Analysis
print("\nPerformance Analysis\n")

if warnings:
    for warning in set(warnings):
        print(f"⚠️ {warning}")
else:
    print("✅ Query looks optimized.")




Execution Plan

{'id': 1, 'select_type': 'SIMPLE', 'table': 'walmart_sales', 'partitions': None, 'type': 'ALL', 'possible_keys': None, 'key': None, 'key_len': None, 'ref': None, 'rows': 9927, 'filtered': 10.0, 'Extra': 'Using where'}

Query Metrics

Table: walmart_sales
Access Type: ALL
Rows Scanned: 9927
Index Used: None
Extra Info: Using where
--------------------------------------------------

Performance Analysis

⚠️ No index used on 'walmart_sales'.
⚠️ Full table scan detected on 'walmart_sales'. Consider creating an index.


Feature-3 (Predict Runtime)

In [12]:
data=pd.read_csv("C:\\Users\\MyPC\\Desktop\\AI Based SQL Query\\query_performance_dataset.csv")
data.shape

(1076, 5)

In [13]:
le = LabelEncoder()
data['access_type'] = le.fit_transform(data['access_type'])

In [14]:
X = data.drop('execution_time_ms', axis=1)
y = data['execution_time_ms']

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [16]:
model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [17]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R²  :", r2)
print(f"Model Accuracy: {r2 * 100:.2f}%")

MAE : 0.32005685567855835
MSE : 0.45103952288627625
RMSE: 0.6715947609133622
R²  : 0.9979535341262817
Model Accuracy: 99.80%


In [ ]:
def predict_runtime(explain_plan, query):

    row = explain_plan["plan"][0]

    features = {
        "query_length": len(query),
        "rows_scanned": row.get("rows", 0),
        "access_type": 1 if row.get("type") == "ALL" else 0,
        "index_used": 0 if row.get("key") is None else 1
    }

    X = pd.DataFrame([features])

    prediction = model.predict(X)

    return float(prediction[0])

Feature-4 (SQL Query Rewriter)

In [ ]:

class SQLQueryRewriter:

    def __init__(self, query):
        self.query = query

    def optimize_select_star(self):

        if "SELECT *" in self.query.upper():

            self.query = re.sub(
                r"SELECT\s+\*",
                "SELECT <SPECIFY_COLUMNS>",
                self.query,
                flags=re.IGNORECASE
            )

    def optimize_year_function(self):

        pattern = r"YEAR\s*\(\s*(\w+)\s*\)\s*=\s*(\d{4})"

        match = re.search(
            pattern,
            self.query,
            re.IGNORECASE
        )

        if match:

            column = match.group(1)
            year = match.group(2)

            replacement = (
                f"{column} >= '{year}-01-01' "
                f"AND {column} < '{int(year)+1}-01-01'"
            )

            self.query = re.sub(
                pattern,
                replacement,
                self.query,
                flags=re.IGNORECASE
            )

    def optimize_order_by_rand(self):

        self.query = re.sub(
            r"ORDER\s+BY\s+RAND\s*\(\s*\)",
            "-- Avoid ORDER BY RAND() on large tables",
            self.query,
            flags=re.IGNORECASE
        )

    def optimize_not_in(self):

        self.query = re.sub(
            r"NOT\s+IN",
            "NOT EXISTS",
            self.query,
            flags=re.IGNORECASE
        )

    def rewrite(self):

        self.optimize_select_star()

        self.optimize_year_function()

        self.optimize_order_by_rand()

        self.optimize_not_in()

        return self.query

In [20]:
query = """
SELECT *
FROM walmart_sales
WHERE YEAR(Date)=2023;
"""

rewriter = SQLQueryRewriter(query)

optimized_query = rewriter.rewrite()

print(optimized_query)


SELECT <SPECIFY_COLUMNS>
FROM walmart_sales
WHERE Date >= '2023-01-01' AND Date < '2024-01-01';



In [21]:
class SQLQueryOptimizer:

    def __init__(self, query):
        self.query = query
        self.suggestions = []

    def generate_suggestions(self):

        query_upper = self.query.upper()

        if "SELECT *" in query_upper:
            self.suggestions.append(
                "Replace SELECT * with specific columns."
            )

        if "YEAR(" in query_upper:
            self.suggestions.append(
                "Avoid YEAR() on indexed columns."
            )

        if "ORDER BY RAND()" in query_upper:
            self.suggestions.append(
                "Avoid ORDER BY RAND() for large tables."
            )

        if "NOT IN" in query_upper:
            self.suggestions.append(
                "Consider NOT EXISTS instead of NOT IN."
            )

        return self.suggestions

In [ ]:
query = """
SELECT *
FROM walmart_sales
WHERE YEAR(Date)=2023
ORDER BY RAND();
"""

optimizer = SQLQueryOptimizer(query)

print("Recommendations:")

for rec in optimizer.generate_suggestions():
    print("-", rec)

rewriter = SQLQueryRewriter(query)

print("\nOptimized Query:\n")

print(rewriter.rewrite())


Recommendations:
- Replace SELECT * with specific columns.
- Avoid YEAR() on indexed columns.
- Avoid ORDER BY RAND() for large tables.

Optimized Query:


SELECT <SPECIFY_COLUMNS>
FROM walmart_sales
WHERE Date >= '2023-01-01' AND Date < '2024-01-01'
-- Avoid ORDER BY RAND() on large tables;



Feature-5 (Ai Sql Optimizer)

In [23]:
def ai_sql_optimizer(query, explain_output):

    prompt = f"""
You are a Senior Database Performance Engineer.

SQL QUERY:
{query}

MYSQL EXPLAIN OUTPUT:
{explain_output}

Analyze the query and provide:

1. Performance Issues
2. Root Cause of Slow Performance
3. Index Recommendations
4. Query Rewrite Suggestions
5. Optimized SQL Query
6. Performance Score (0-100)

Return the response in this format:

PERFORMANCE ISSUES:
- ...

ROOT CAUSE:
- ...

INDEX RECOMMENDATIONS:
- ...

QUERY REWRITE SUGGESTIONS:
- ...

OPTIMIZED QUERY:
SELECT ...

PERFORMANCE SCORE:
...
"""

    response = llm.invoke(prompt)
    return response.content

In [24]:
query = """
SELECT *
FROM walmart_sales
WHERE Branch = 'A';
"""

explain_output = {
    'id': 1,
    'select_type': 'SIMPLE',
    'table': 'walmart_sales',
    'type': 'ALL',
    'possible_keys': None,
    'key': None,
    'rows': 9927,
    'Extra': 'Using where'
}

report = ai_sql_optimizer(
    query=query,
    explain_output=explain_output
)

print(report)

PERFORMANCE ISSUES:
-   **Full Table Scan**: The `type: ALL` in the EXPLAIN output indicates that MySQL is performing a full table scan on `walmart_sales`. This means it reads every single row (9927 rows) from the table to find those matching the `WHERE` clause.
-   **Inefficient I/O**: Reading all rows from disk (or memory) is highly inefficient, especially as the table grows.
-   **CPU Overhead**: The database has to evaluate the `Branch = 'A'` condition for every single row, consuming unnecessary CPU cycles.
-   **Scalability**: Performance will degrade linearly as the `walmart_sales` table increases in size. A table with 9927 rows might be acceptable for a full scan in some cases, but it's a critical bottleneck for larger datasets or high-frequency queries.
-   **No Index Utilization**: `possible_keys: None` and `key: None` confirm that no index was considered or used, leading to the full table scan.

ROOT CAUSE:
-   **Missing Index**: The primary root cause is the absence of an ap

In [25]:
def optimize_sql(query):

    # Feature 1
    issues = SQLAnalyzer(query).analyze()

    # Feature 2
    plan = get_explain_plan(query)

    # Feature 3
    runtime_prediction = predict_runtime(plan, query)

    # Feature 4
    optimized_query = SQLQueryRewriter(query).rewrite()

    # Feature 5
    ai_report = ai_sql_optimizer(
        query=query,
        explain_output=plan
    )

    return {
        "issues": issues,
        "execution_plan": plan,
        "predicted_runtime": runtime_prediction,
        "optimized_query": optimized_query,
        "ai_report": ai_report
    }

In [ ]:
query = """
SELECT *
FROM walmart_sales ws1
WHERE ws1.profit_margin > (
    SELECT AVG(ws2.profit_margin)
    FROM walmart_sales ws2
    WHERE ws2.City = ws1.City
)
AND ws1.Branch IN (
    SELECT Branch
    FROM walmart_sales
    WHERE quantity > (
        SELECT AVG(quantity)
        FROM walmart_sales
    )
)
AND YEAR(ws1.date) = 2019
AND MONTH(ws1.date) = 3
AND ws1.category LIKE '%Elect%'
AND ws1.payment_method IN (
    SELECT payment_method
    FROM walmart_sales
    WHERE rating > 4
)
ORDER BY ws1.profit_margin DESC,
         ws1.quantity DESC;
"""


In [30]:
def print_report(result):

    print("\n" + "="*80)
    print("SQL QUERY OPTIMIZATION REPORT")
    print("="*80)

    print("\nPERFORMANCE ISSUES")
    print("-"*80)

    for issue in result["issues"]:
        print(f"• {issue}")

    print("\nEXECUTION PLAN WARNINGS")
    print("-"*80)

    for warning in result["execution_plan"]["warnings"]:
        print(f"• {warning}")

    print("\nPREDICTED RUNTIME")
    print("-"*80)
    print(f"{result['predicted_runtime']:.2f} ms")

    print("\nOPTIMIZED QUERY")
    print("-"*80)
    print(result["optimized_query"])

    print("\nAI ANALYSIS REPORT")
    print("-"*80)
    print(result["ai_report"])

    print("\n" + "="*80)

In [31]:
result = optimize_sql(query)

print_report(result)


SQL QUERY OPTIMIZATION REPORT

PERFORMANCE ISSUES
--------------------------------------------------------------------------------
• Avoid using SELECT *. Specify only required columns.
• Function 'YEAR' used in WHERE clause may prevent index usage.
• Function 'MONTH' used in WHERE clause may prevent index usage.
• Subquery detected. Consider replacing with JOIN if appropriate.

EXECUTION PLAN WARNINGS
--------------------------------------------------------------------------------
• Temporary table created. Query may be optimized.
• Full table scan detected on 'ws1'. Consider creating an index.
• No index used on 'ws1'.
• No index used on 'walmart_sales'.
• No index used on 'ws2'.
• Full table scan detected on 'ws2'. Consider creating an index.
• Filesort detected. Consider indexing ORDER BY columns.
• Full table scan detected on 'walmart_sales'. Consider creating an index.

PREDICTED RUNTIME
--------------------------------------------------------------------------------
28.46 ms

O